# Vector Store Debug Notebook

This notebook helps debug RAG retrieval issues by:
1. Querying vector stores and examining similarity scores
2. Testing similarity against different metadata fields (title, project name, etc.)
3. Comparing chunk-level vs project-level retrieval
4. **Debugging the Query Processor (LLM-based entity extraction)**

In [3]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
from pathlib import Path

from rag.vectorstore import VectorStore
from rag.project_vectorstore import ProjectVectorStore
from rag.embeddings import EmbeddingManager

print("Imports successful!")

/Users/bruengw/Library/CloudStorage/OneDrive-AbbVieInc(O365)/Documents/confluence_rag/.conf_rag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful!


## Configuration

In [4]:
# === CONFIGURATION ===

# Which vector store to query: 'chunks' or 'projects'
STORE_TYPE = 'chunks'  # Change to 'projects' to query project store

# Number of results to retrieve
TOP_K = 20

# Test query
QUERY = "What is ALFA?"

# Paths
CHUNK_STORE_PATH = "../Data_Storage/vector_db"
PROJECT_STORE_PATH = "../Data_Storage/project_vector_db"

## Load Vector Stores

In [5]:
# Initialize embedding manager
embedding_manager = EmbeddingManager()
print(f"Embedding model: {embedding_manager.model_name}")
print(f"Embedding dimension: {embedding_manager.embedding_dimension}")

2026-03-12 09:21:12.498 | INFO     | rag.embeddings:__init__:26 - Loading embedding model: all-MiniLM-L6-v2
2026-03-12 09:21:16.476 | INFO     | rag.embeddings:__init__:29 - Embedding model loaded. Dimension: 384


Embedding model: all-MiniLM-L6-v2
Embedding dimension: 384


In [6]:
# Load chunk vector store
chunk_store = VectorStore(persist_directory=CHUNK_STORE_PATH)
print(f"Chunk store loaded: {chunk_store.count()} chunks")
print(f"Embeddings shape: {chunk_store.embeddings.shape}")

2026-03-12 09:21:16.501 | INFO     | rag.vectorstore:__init__:44 - Initializing VectorStore at: ../Data_Storage/vector_db
2026-03-12 09:21:16.571 | INFO     | rag.vectorstore:_load_or_initialize:71 - Loaded 13879 documents from disk
2026-03-12 09:21:16.572 | INFO     | rag.vectorstore:__init__:52 - Vector store initialized with collection: confluence_docs
2026-03-12 09:21:16.573 | DEBUG    | rag.vectorstore:count:429 - Vector store contains 13879 documents


Chunk store loaded: 13879 chunks
Embeddings shape: (13879, 384)


In [7]:
# Load project vector store
project_store = ProjectVectorStore(persist_directory=PROJECT_STORE_PATH)
print(f"Project store loaded: {project_store.count()} projects")
print(f"Embeddings shape: {project_store.embeddings.shape}")
print(f"\nProjects in store:")
for name in project_store.get_project_names():
    print(f"  - {name}")

2026-03-12 09:21:16.615 | INFO     | rag.project_vectorstore:_load_or_initialize:109 - Loaded 143 projects from disk
2026-03-12 09:21:16.615 | INFO     | rag.project_vectorstore:__init__:86 - Initialized ProjectVectorStore (dir=../Data_Storage/project_vector_db, model=all-MiniLM-L6-v2)


Project store loaded: 143 projects
Embeddings shape: (143, 384)

Projects in store:
  - ASML/GSML
  - ATLAS
  - ATOMS
  - ATOMS AD Subtyping Analysis
  - ATOMS GEA Subtyping Analysis
  - ATOMS HS LUTI Network Analysis
  - ATOMS HS Subtyping Analysis
  - ATOMS NSCLC Subtyping Analysis
  - Ad-Stock Non-Linear Mixed Modeling
  - Automate and fine-tune Jarvis Dashboard checks
  - BEx Aesthetics Portfolio Management Project
  - CAMPAIGN Project Template
  - CAP - APEX Project
  - CARIS Overall Survival prediction and patient prognostic subtyping
  - CDSO - Continuous Data Review (CDR)
  - CDSO - Operational Insight & Performance (OIP)
  - CDSO - Operations Intelligence Tool (OIT)
  - CDSO Rollup Smartsheet
  - CLOVER
  - CLOVERX Project
  - COMET
  - CRA Pre-Visit Summary
  - CSC - Global Site Survey (GSS)
  - CSC - Microsurvey Support
  - CSM CRM Enhancement - V2.0
  - CSM Pipeline Feature Store
  - Catalyst
  - Catalyst - Fix the confidence interval calculation methodology
  - Catalyst Co

---
## Section 1: Query Vector Store with Detailed Similarity Scores

In [8]:
def query_with_details(query: str, store_type: str = 'chunks', top_k: int = 10):
    """
    Query vector store and return detailed similarity information.
    """
    print(f"{'='*80}")
    print(f"QUERY: {query}")
    print(f"STORE: {store_type}")
    print(f"TOP_K: {top_k}")
    print(f"{'='*80}\n")
    
    # Generate query embedding
    query_embedding = embedding_manager.generate_embedding(query)
    print(f"Query embedding shape: {query_embedding.shape}")
    print(f"Query embedding norm: {np.linalg.norm(query_embedding):.4f}")
    print()
    
    # Select store
    if store_type == 'chunks':
        store = chunk_store
    else:
        store = project_store
    
    embeddings = store.embeddings
    metadatas = store.metadatas
    documents = store.documents
    
    # Compute cosine similarity
    query_norm = query_embedding / (np.linalg.norm(query_embedding) + 1e-10)
    embedding_norms = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-10
    normalized_embeddings = embeddings / embedding_norms
    similarities = np.dot(normalized_embeddings, query_norm)
    
    # Statistics
    print(f"Similarity Statistics:")
    print(f"  Min:  {similarities.min():.6f}")
    print(f"  Max:  {similarities.max():.6f}")
    print(f"  Mean: {similarities.mean():.6f}")
    print(f"  Std:  {similarities.std():.6f}")
    print()
    
    # Get top-k indices
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    # Build results dataframe
    results = []
    for rank, idx in enumerate(top_indices, 1):
        meta = metadatas[idx]
        sim = similarities[idx]
        doc_preview = documents[idx][:200].replace('\n', ' ')
        
        result = {
            'rank': rank,
            'similarity': sim,
            'distance': 1 - sim,
        }
        
        if store_type == 'chunks':
            result.update({
                'title': meta.get('title', 'N/A')[:50],
                'main_project': meta.get('main_project', 'N/A'),
                'chunk_index': meta.get('chunk_index', 'N/A'),
                'depth': meta.get('depth', 'N/A'),
                'content_preview': doc_preview,
            })
        else:
            result.update({
                'main_project': meta.get('main_project', 'N/A'),
                'page_count': meta.get('page_count', 0),
                'content_preview': doc_preview,
            })
        
        results.append(result)
    
    df = pd.DataFrame(results)
    return df, similarities

In [9]:
# Run query against selected store
results_df, all_similarities = query_with_details(QUERY, store_type=STORE_TYPE, top_k=TOP_K)

# Display results
pd.set_option('display.max_colwidth', 100)
results_df

QUERY: What is ALFA?
STORE: chunks
TOP_K: 20



2026-03-12 09:21:37.915 | DEBUG    | rag.embeddings:generate_embedding:46 - Generated embedding for text of length 13


Query embedding shape: (384,)
Query embedding norm: 1.0000

Similarity Statistics:
  Min:  -0.196838
  Max:  0.475895
  Mean: 0.049392
  Std:  0.057935



,rank,similarity,distance,title,main_project,chunk_index,depth,content_preview
0,1,0.475895,0.524105,05. ALFA,N/A,N/A,N/A,ALFA The Anomaly Detection project leverages advanced statistical testing and mixed-effects mod...
1,2,0.450973,0.549027,05. ALFA,RBQM Analytics,N/A,4,ALFA The Anomaly Detection project leverages advanced statistical testing and mixed-effects mod...
2,3,0.450973,0.549027,05. ALFA,N/A,N/A,N/A,ALFA The Anomaly Detection project leverages advanced statistical testing and mixed-effects mod...
3,4,0.450973,0.549027,05. ALFA,RBQM Analytics,N/A,4,ALFA The Anomaly Detection project leverages advanced statistical testing and mixed-effects mod...
4,5,0.450973,0.549027,05. ALFA,RBQM Analytics,N/A,4,ALFA The Anomaly Detection project leverages advanced statistical testing and mixed-effects mod...
5,6,0.450973,0.549027,05. ALFA,N/A,N/A,N/A,ALFA The Anomaly Detection project leverages advanced statistical testing and mixed-effects mod...
6,7,0.380890,0.619110,05.4 Data understanding,N/A,N/A,N/A,"The Database used in ALFA is ""SAM"" (previously LSH was used) Data Dictionary: - Contains all t..."
7,8,0.360184,0.639816,4.5 RBQM Meeting Minutes,N/A,N/A,N/A,V server. 2024 June 20 DE & UI CONNECT: Some of the KRI data coming across have different end da...
8,9,0.333496,0.666504,9.7.3 DSA SS Survey Links,N/A,N/A,N/A,ALFA ATLAS CAMPAIGN ATOMS Catalyst Clinical Analytics Landing Page CLOVER COM Dashboard COMET Co...
9,10,0.333496,0.666504,9.7.3 DSA SS Survey Links,DSA Centralized App,N/A,5,ALFA ATLAS CAMPAIGN ATOMS Catalyst Clinical Analytics Landing Page CLOVER COM Dashboard COMET Co...


In [ ]:
# Visualize similarity distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram of all similarities
axes[0].hist(all_similarities, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Similarity Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of All Similarity Scores')
axes[0].axvline(x=all_similarities.mean(), color='red', linestyle='--', label=f'Mean: {all_similarities.mean():.4f}')
axes[0].legend()

# Top-k similarities
axes[1].bar(range(len(results_df)), results_df['similarity'], color='steelblue')
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Similarity Score')
axes[1].set_title(f'Top {TOP_K} Similarity Scores')
axes[1].set_xticks(range(len(results_df)))
axes[1].set_xticklabels(range(1, len(results_df) + 1))

plt.tight_layout()
plt.show()

---
## Section 2: Compare Similarity Against Different Metadata Fields

Test if searching against page titles or project names yields better results than content.

In [10]:
def compute_similarity_to_field(query: str, field_name: str, store_type: str = 'chunks'):
    """
    Compute similarity between query and a specific metadata field.
    """
    # Select store
    if store_type == 'chunks':
        metadatas = chunk_store.metadatas
    else:
        metadatas = project_store.metadatas
    
    # Extract field values
    field_values = [meta.get(field_name, '') or '' for meta in metadatas]
    unique_values = list(set(v for v in field_values if v))
    
    print(f"Field: {field_name}")
    print(f"Unique values: {len(unique_values)}")
    print(f"Sample values: {unique_values[:10]}")
    print()
    
    # Generate query embedding
    query_embedding = embedding_manager.generate_embedding(query)
    
    # Generate embeddings for unique field values
    if not unique_values:
        print("No values found for this field!")
        return None
    
    print(f"Generating embeddings for {len(unique_values)} unique {field_name} values...")
    field_embeddings = embedding_manager.generate_embeddings(unique_values)
    
    # Compute similarities
    query_norm = query_embedding / (np.linalg.norm(query_embedding) + 1e-10)
    field_norms = np.linalg.norm(field_embeddings, axis=1, keepdims=True) + 1e-10
    normalized_field_embeddings = field_embeddings / field_norms
    similarities = np.dot(normalized_field_embeddings, query_norm)
    
    # Create results dataframe
    results = []
    for i, (value, sim) in enumerate(zip(unique_values, similarities)):
        results.append({
            field_name: value,
            'similarity': sim,
        })
    
    df = pd.DataFrame(results).sort_values('similarity', ascending=False).reset_index(drop=True)
    df.index = df.index + 1  # 1-indexed rank
    df.index.name = 'rank'
    
    return df

In [11]:
# Test query against page TITLES
print("=" * 80)
print(f"COMPARING QUERY TO PAGE TITLES")
print(f"Query: {QUERY}")
print("=" * 80)
print()

title_similarities = compute_similarity_to_field(QUERY, 'title', store_type='chunks')
if title_similarities is not None:
    print("\nTop 20 most similar titles:")
    display(title_similarities.head(20))

COMPARING QUERY TO PAGE TITLES
Query: What is ALFA?

Field: title
Unique values: 1559
Sample values: ['16.3 RHEUM Data Sources', '12. RISE-AI - EM - Transition Plan', '14.X.1.2. Catalyst - Analysis', 'Liquid Biopsy Model Development', 'BA Projects - 2.1.2 File Directory', '6.2 Clover 2.0 Project - DE Requirements Gathering', 'BA Projects - 2.4 Exploratory Analysis', '0.3 Medical Affairs Notes', '2. CTI RA - Data Engineering Execution Plan', '8.2. Catalyst - Default Risk']



2026-03-12 09:21:56.223 | DEBUG    | rag.embeddings:generate_embedding:46 - Generated embedding for text of length 13
2026-03-12 09:21:56.224 | INFO     | rag.embeddings:generate_embeddings:67 - Generating embeddings for 1559 texts


Generating embeddings for 1559 unique title values...


Batches: 100%|██████████| 49/49 [00:01<00:00, 37.73it/s]
2026-03-12 09:21:57.534 | INFO     | rag.embeddings:generate_embeddings:76 - Generated 1559 embeddings



Top 20 most similar titles:


,title,similarity
rank,,
1,05. ALFA,0.602259
2,CLOVER,0.343742
3,Clover 1.0 Product Evaluation,0.296707
4,ATLAS,0.291884
5,23.3 CLOVER Data Sources,0.287000
6,23.4 CLOVER Documentation,0.274599
7,Arch,0.266177
8,Catalyst,0.261715
9,ATLAS_,0.258483


In [12]:
# Test query against MAIN_PROJECT names
print("=" * 80)
print(f"COMPARING QUERY TO MAIN_PROJECT NAMES")
print(f"Query: {QUERY}")
print("=" * 80)
print()

project_similarities = compute_similarity_to_field(QUERY, 'main_project', store_type='chunks')
if project_similarities is not None:
    print("\nAll projects ranked by similarity:")
    display(project_similarities)

COMPARING QUERY TO MAIN_PROJECT NAMES
Query: What is ALFA?

Field: main_project
Unique values: 143
Sample values: ['Exploratory data analysis to generate evidence', 'CAMPAIGN Project Template', 'GCA Longitudinal Biomarker Analysis For UPA Response', 'RBQM Analytics', 'IBD RT project', 'Publication Efficacy Extraction', 'Patient Support Program Vyalev', 'Global Site Survey', 'CSC - Global Site Survey (GSS)', 'Design and Build the Data Ingestion : Feature store Layer']



2026-03-12 09:22:02.797 | DEBUG    | rag.embeddings:generate_embedding:46 - Generated embedding for text of length 13
2026-03-12 09:22:02.797 | INFO     | rag.embeddings:generate_embeddings:67 - Generating embeddings for 143 texts


Generating embeddings for 143 unique main_project values...


Batches: 100%|██████████| 5/5 [00:00<00:00, 23.55it/s]
2026-03-12 09:22:03.017 | INFO     | rag.embeddings:generate_embeddings:76 - Generated 143 embeddings



All projects ranked by similarity:


,main_project,similarity
rank,,
1,CLOVER,0.343742
2,ATLAS,0.291884
3,IBD RT project,0.290463
4,Catalyst,0.261715
5,Vendor Health Platform,0.241714
...,...,...
139,CSC - Global Site Survey (GSS),-0.051576
140,CDSO Rollup Smartsheet,-0.068690
141,CSC - Microsurvey Support,-0.072433


---
## Section 3: Side-by-Side Comparison of Content vs Title Similarity

In [ ]:
def compare_content_vs_title_retrieval(query: str, top_k: int = 10):
    """
    Compare what gets retrieved when searching content embeddings vs title embeddings.
    """
    print(f"Query: {query}")
    print(f"="*80)
    
    # Generate query embedding
    query_embedding = embedding_manager.generate_embedding(query)
    query_norm = query_embedding / (np.linalg.norm(query_embedding) + 1e-10)
    
    # --- Content-based similarity (existing embeddings) ---
    content_embeddings = chunk_store.embeddings
    content_norms = np.linalg.norm(content_embeddings, axis=1, keepdims=True) + 1e-10
    normalized_content = content_embeddings / content_norms
    content_similarities = np.dot(normalized_content, query_norm)
    
    # --- Title-based similarity (compute on the fly) ---
    titles = [meta.get('title', '') or '' for meta in chunk_store.metadatas]
    print(f"Generating title embeddings for {len(titles)} chunks...")
    title_embeddings = embedding_manager.generate_embeddings(titles)
    title_norms = np.linalg.norm(title_embeddings, axis=1, keepdims=True) + 1e-10
    normalized_titles = title_embeddings / title_norms
    title_similarities = np.dot(normalized_titles, query_norm)
    
    # --- Combined similarity ---
    combined_similarities = 0.5 * content_similarities + 0.5 * title_similarities
    
    # Get top-k for each method
    content_top = np.argsort(content_similarities)[::-1][:top_k]
    title_top = np.argsort(title_similarities)[::-1][:top_k]
    combined_top = np.argsort(combined_similarities)[::-1][:top_k]
    
    # Build comparison dataframe
    comparison = []
    for rank in range(top_k):
        content_idx = content_top[rank]
        title_idx = title_top[rank]
        combined_idx = combined_top[rank]
        
        comparison.append({
            'rank': rank + 1,
            'content_title': chunk_store.metadatas[content_idx].get('title', '')[:40],
            'content_project': chunk_store.metadatas[content_idx].get('main_project', ''),
            'content_sim': f"{content_similarities[content_idx]:.4f}",
            'title_title': chunk_store.metadatas[title_idx].get('title', '')[:40],
            'title_project': chunk_store.metadatas[title_idx].get('main_project', ''),
            'title_sim': f"{title_similarities[title_idx]:.4f}",
            'combined_title': chunk_store.metadatas[combined_idx].get('title', '')[:40],
            'combined_project': chunk_store.metadatas[combined_idx].get('main_project', ''),
            'combined_sim': f"{combined_similarities[combined_idx]:.4f}",
        })
    
    return pd.DataFrame(comparison)

In [ ]:
# Compare retrieval methods
comparison_df = compare_content_vs_title_retrieval(QUERY, top_k=10)

print("\nContent-based retrieval:")
display(comparison_df[['rank', 'content_title', 'content_project', 'content_sim']])

print("\nTitle-based retrieval:")
display(comparison_df[['rank', 'title_title', 'title_project', 'title_sim']])

print("\nCombined (50/50) retrieval:")
display(comparison_df[['rank', 'combined_title', 'combined_project', 'combined_sim']])

---
## Section 4: Investigate Specific Project/Document

In [ ]:
def find_chunks_by_project(project_name: str, query: str = None):
    """
    Find all chunks belonging to a specific project and optionally compute similarity.
    """
    matching_indices = []
    for i, meta in enumerate(chunk_store.metadatas):
        if meta.get('main_project', '').lower() == project_name.lower():
            matching_indices.append(i)
    
    print(f"Found {len(matching_indices)} chunks for project: {project_name}")
    
    if not matching_indices:
        # Try partial match
        for i, meta in enumerate(chunk_store.metadatas):
            if project_name.lower() in meta.get('main_project', '').lower():
                matching_indices.append(i)
        print(f"Partial match found {len(matching_indices)} chunks")
    
    if not matching_indices:
        return None
    
    # Compute similarity if query provided
    if query:
        query_embedding = embedding_manager.generate_embedding(query)
        query_norm = query_embedding / (np.linalg.norm(query_embedding) + 1e-10)
        
        embeddings = chunk_store.embeddings[matching_indices]
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-10
        normalized = embeddings / norms
        similarities = np.dot(normalized, query_norm)
    else:
        similarities = [None] * len(matching_indices)
    
    # Build results
    results = []
    for i, idx in enumerate(matching_indices):
        meta = chunk_store.metadatas[idx]
        results.append({
            'chunk_index': meta.get('chunk_index', 'N/A'),
            'title': meta.get('title', 'N/A')[:50],
            'depth': meta.get('depth', 'N/A'),
            'similarity': similarities[i] if query else None,
            'content_preview': chunk_store.documents[idx][:150].replace('\n', ' '),
        })
    
    df = pd.DataFrame(results)
    if query:
        df = df.sort_values('similarity', ascending=False)
    return df

In [ ]:
# Find chunks for a specific project
TARGET_PROJECT = "ATLAS"  # Change this to investigate a specific project

project_chunks = find_chunks_by_project(TARGET_PROJECT, query=QUERY)
if project_chunks is not None:
    print(f"\nChunks for '{TARGET_PROJECT}' ranked by similarity to '{QUERY}':")
    display(project_chunks)

---
## Section 5: Check for Embedding Issues

In [ ]:
def diagnose_embeddings(store_type: str = 'chunks'):
    """Check for common embedding issues."""
    
    if store_type == 'chunks':
        embeddings = chunk_store.embeddings
        name = "Chunk Store"
    else:
        embeddings = project_store.embeddings
        name = "Project Store"
    
    print(f"Diagnosing {name}")
    print("=" * 50)
    print(f"Shape: {embeddings.shape}")
    print(f"Dtype: {embeddings.dtype}")
    print()
    
    # Check for NaN/Inf
    nan_count = np.isnan(embeddings).sum()
    inf_count = np.isinf(embeddings).sum()
    print(f"NaN values: {nan_count}")
    print(f"Inf values: {inf_count}")
    print()
    
    # Check norms
    norms = np.linalg.norm(embeddings, axis=1)
    print(f"Embedding norms:")
    print(f"  Min:  {norms.min():.6f}")
    print(f"  Max:  {norms.max():.6f}")
    print(f"  Mean: {norms.mean():.6f}")
    print(f"  Std:  {norms.std():.6f}")
    print()
    
    # Check for zero vectors
    zero_vectors = (norms < 1e-6).sum()
    print(f"Zero/near-zero vectors: {zero_vectors}")
    
    # Check for duplicate embeddings
    unique_count = len(np.unique(embeddings, axis=0))
    duplicate_count = len(embeddings) - unique_count
    print(f"Duplicate embeddings: {duplicate_count}")
    print()
    
    # Sample pairwise similarities
    n_samples = min(100, len(embeddings))
    sample_indices = np.random.choice(len(embeddings), n_samples, replace=False)
    sample_embeddings = embeddings[sample_indices]
    
    # Normalize
    sample_norms = np.linalg.norm(sample_embeddings, axis=1, keepdims=True) + 1e-10
    normalized_samples = sample_embeddings / sample_norms
    
    # Pairwise similarity matrix
    similarity_matrix = np.dot(normalized_samples, normalized_samples.T)
    
    # Get upper triangle (excluding diagonal)
    upper_tri = similarity_matrix[np.triu_indices(n_samples, k=1)]
    
    print(f"Pairwise similarities (sample of {n_samples}):")
    print(f"  Min:  {upper_tri.min():.6f}")
    print(f"  Max:  {upper_tri.max():.6f}")
    print(f"  Mean: {upper_tri.mean():.6f}")
    print(f"  Std:  {upper_tri.std():.6f}")
    
    if upper_tri.std() < 0.01:
        print("\n⚠️  WARNING: Very low variance in similarities - embeddings may be too similar!")
    
    return norms, similarity_matrix

In [ ]:
# Diagnose chunk store
chunk_norms, chunk_sim_matrix = diagnose_embeddings('chunks')

In [ ]:
# Diagnose project store
project_norms, project_sim_matrix = diagnose_embeddings('projects')

---
## Section 6: Interactive Query Testing

In [ ]:
# Quick test function - change query and run
def quick_test(query: str, top_k: int = 5):
    """Quick test against both stores."""
    print(f"Query: {query}")
    print("="*60)
    
    # Project store
    print("\n--- PROJECT STORE ---")
    results = project_store.query_projects(query, n_results=top_k)
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r['main_project']}: sim={r['similarity']:.4f}")
    
    # Chunk store
    print("\n--- CHUNK STORE ---")
    results = chunk_store.query(query, n_results=top_k)
    for i in range(len(results['documents'])):
        meta = results['metadatas'][i]
        dist = results['distances'][i]
        print(f"  {i+1}. {meta.get('title', 'N/A')[:40]}")
        print(f"      project={meta.get('main_project', 'N/A')}, dist={dist:.4f}")

In [ ]:
# Test various queries
test_queries = [
    "What is ALFA?",
    "ALFA project",
    "Tell me about the ALFA project",
    "data pipeline",
    "machine learning",
]

for q in test_queries:
    quick_test(q, top_k=3)
    print("\n")

---
## Section 7: Debug Query Processor (LLM-based Entity Extraction)

Test whether the QueryProcessor correctly extracts project names, people, dates, and other entities from queries.

In [13]:
# Import QueryProcessor
from rag.query_processor import QueryProcessor, ProcessedQuery

# Initialize with LLM (set use_llm=False to test regex fallback)
USE_LLM = True
query_processor = QueryProcessor(use_llm=USE_LLM, use_few_shot=True)

print(f"QueryProcessor initialized:")
print(f"  LLM enabled: {query_processor.use_llm}")
print(f"  Client available: {query_processor.iliad_client is not None}")
print(f"  Model: {query_processor.model}")

2026-03-12 09:22:22.077 | INFO     | iliad.client:__init__:135 - Initialized IliadClient with base URL: https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/chat/gpt-4o
2026-03-12 09:22:22.088 | INFO     | rag.query_processor:__init__:220 - Initialized Iliad client from environment
2026-03-12 09:22:22.093 | INFO     | rag.query_processor:__init__:225 - QueryProcessor initialized (LLM: True, Model: gpt-4o-mini-global, Few-shot: True)


QueryProcessor initialized:
  LLM enabled: True
  Client available: True
  Model: gpt-4o-mini-global


In [14]:
def display_processed_query(result: ProcessedQuery):
    """Display processed query results in a formatted way."""
    print("=" * 80)
    print(f"ORIGINAL QUERY: {result.original_query}")
    print("=" * 80)
    print()
    
    print(f"Cleaned Query:    {result.cleaned_query}")
    print(f"Query Intent:     {result.query_intent}")
    print(f"Is Comparative:   {result.is_comparative}")
    print(f"Confidence:       {result.confidence:.2f}")
    print()
    
    print("EXTRACTED ENTITIES:")
    print("-" * 40)
    
    print(f"  Keywords ({len(result.keywords)}):")
    for kw in result.keywords:
        print(f"    - {kw}")
    
    print(f"\n  Project Names ({len(result.potential_project_names)}):")
    for proj in result.potential_project_names:
        print(f"    - {proj}")
        # Check if project exists in store
        if proj.upper() in [p.upper() for p in project_store.get_project_names()]:
            print(f"      ✓ Found in project store")
        else:
            print(f"      ✗ NOT in project store")
    
    print(f"\n  Person Names ({len(result.potential_person_names)}):")
    for person in result.potential_person_names:
        print(f"    - {person}")
    
    print(f"\n  Dates ({len(result.dates)}):")
    for date in result.dates:
        print(f"    - {date}")
    
    print(f"\n  Technologies ({len(result.technologies)}):")
    for tech in result.technologies:
        print(f"    - {tech}")
    
    if result.is_comparative:
        print(f"\n  Comparative Entities:")
        for entity in result.comparative_entities:
            print(f"    - {entity}")
    
    print()
    return result

In [20]:
# Test single query
TEST_QUERY = "What is the ALFA project and who is working on it? Does Kunehi work on it? What is the ALFA project and who is working on it? Does Kunehi work on it?"

result = query_processor.process_query(TEST_QUERY)
display_processed_query(result)

2026-03-12 09:33:19.098 | INFO     | rag.query_processor:process_query:240 - Processing query: 'What is the ALFA project and who is working on it? Does Kunehi work on it? What is the ALFA project and who is working on it? Does Kunehi work on it?'
2026-03-12 09:33:19.099 | DEBUG    | iliad.client:chat:245 - Chat request with 1 messages
2026-03-12 09:33:19.100 | DEBUG    | iliad.client:_make_request:190 - Request attempt 1/3: POST https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/chat/gpt-4o
2026-03-12 09:33:21.765 | DEBUG    | iliad.client:chat:249 - Chat response received: 508 chars
2026-03-12 09:33:21.850 | INFO     | rag.query_processor:process_query:247 - LLM extraction successful - Keywords: 5, Projects: 1, People: 1


ORIGINAL QUERY: What is the ALFA project and who is working on it? Does Kunehi work on it? What is the ALFA project and who is working on it? Does Kunehi work on it?

Cleaned Query:    ALFA project who working on it Kunehi work on it
Query Intent:     informational
Is Comparative:   False
Confidence:       0.90

EXTRACTED ENTITIES:
----------------------------------------
  Keywords (5):
    - ALFA
    - project
    - working
    - Kunehi
    - work

  Project Names (1):
    - ALFA
      ✗ NOT in project store

  Person Names (1):
    - Kunehi

  Dates (0):

  Technologies (0):



ProcessedQuery(original_query='What is the ALFA project and who is working on it? Does Kunehi work on it? What is the ALFA project and who is working on it? Does Kunehi work on it?', cleaned_query='ALFA project who working on it Kunehi work on it', keywords=['ALFA', 'project', 'working', 'Kunehi', 'work'], lemmatized_keywords=['ALFA', 'project', 'working', 'Kunehi', 'work'], potential_project_names=['ALFA'], potential_person_names=['Kunehi'], is_comparative=False, comparative_entities=[], dates=[], technologies=[], query_intent='informational', confidence=0.9)

In [23]:
# Test multiple queries to evaluate project name extraction
test_queries = [
    "What is global lab grading?",
    "Tell me about the ATLAS project",
    "Compare finder to clover methodologies",
    "What projects did John Smith work on in 2024?",
    "How does the RBQM system use machine learning?",
    "What are the data sources for catalyst?",
    "List all DSA projects using R and AWS",
    "Describe the RISE-AI architecture",
]

print("Testing Query Processor on multiple queries...")
print("=" * 80)

results_summary = []
for query in test_queries:
    result = query_processor.process_query(query)
    
    # Check how many extracted projects are in the store
    found_projects = []
    missing_projects = []
    for proj in result.potential_project_names:
        if proj.upper() in [p.upper() for p in project_store.get_project_names()]:
            found_projects.append(proj)
        else:
            missing_projects.append(proj)
    
    results_summary.append({
        'query': query[:50] + '...' if len(query) > 50 else query,
        'intent': result.query_intent,
        'keywords': len(result.keywords),
        'projects_extracted': result.potential_project_names,
        'projects_found': found_projects,
        'projects_missing': missing_projects,
        'people': result.potential_person_names,
        'dates': result.dates,
        'technologies': result.technologies,
        'is_comparative': result.is_comparative,
        'confidence': result.confidence,
    })

# Display as DataFrame
summary_df = pd.DataFrame(results_summary)
pd.set_option('display.max_colwidth', None)
display(summary_df[['query', 'intent', 'projects_extracted', 'projects_found', 'projects_missing', 'people']])

2026-03-12 09:36:09.304 | INFO     | rag.query_processor:process_query:240 - Processing query: 'What is global lab grading?'
2026-03-12 09:36:09.307 | DEBUG    | iliad.client:chat:245 - Chat request with 1 messages
2026-03-12 09:36:09.308 | DEBUG    | iliad.client:_make_request:190 - Request attempt 1/3: POST https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/chat/gpt-4o


Testing Query Processor on multiple queries...


2026-03-12 09:36:10.799 | DEBUG    | iliad.client:chat:249 - Chat response received: 444 chars
2026-03-12 09:36:10.801 | INFO     | rag.query_processor:process_query:247 - LLM extraction successful - Keywords: 3, Projects: 0, People: 0
2026-03-12 09:36:10.802 | INFO     | rag.query_processor:process_query:240 - Processing query: 'Tell me about the ATLAS project'
2026-03-12 09:36:10.802 | DEBUG    | iliad.client:chat:245 - Chat request with 1 messages
2026-03-12 09:36:10.804 | DEBUG    | iliad.client:_make_request:190 - Request attempt 1/3: POST https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/chat/gpt-4o
2026-03-12 09:36:11.835 | DEBUG    | iliad.client:chat:249 - Chat response received: 439 chars
2026-03-12 09:36:11.836 | INFO     | rag.query_processor:process_query:247 - LLM extraction successful - Keywords: 2, Projects: 1, People: 0
2026-03-12 09:36:11.837 | INFO     | rag.query_processor:process_query:240 - Processing query: 'Compare finder to clover methodologies'
2026-03-12

,query,intent,projects_extracted,projects_found,projects_missing,people
0,What is global lab grading?,informational,[],[],[],[]
1,Tell me about the ATLAS project,informational,[ATLAS],[ATLAS],[],[]
2,Compare finder to clover methodologies,comparison,[],[],[],[]
3,What projects did John Smith work on in 2024?,informational,[],[],[],[John Smith]
4,How does the RBQM system use machine learning?,informational,[RBQM],[RBQM],[],[]
5,What are the data sources for catalyst?,informational,[catalyst],[catalyst],[],[]
6,List all DSA projects using R and AWS,listing,[DSA],[],[DSA],[]
7,Describe the RISE-AI architecture,informational,[RISE-AI],[],[RISE-AI],[]


### Compare LLM vs Regex Extraction

Run the same queries through both methods to see the difference.

In [26]:
# Compare LLM vs Regex extraction
llm_processor = QueryProcessor(use_llm=True, use_few_shot=True)
regex_processor = QueryProcessor(use_llm=False)

comparison_queries = [
    # "What is ALFA?",
    # "Compare CLOVER to PASSPORT",
    # "What projects use Python for machine learning?",
    # "Tell me about work John Smith did on ATLAS in 2024",
    "Compare finder to clover methodologies"
]

print("Comparing LLM vs Regex Extraction")
print("=" * 80)

for query in comparison_queries:
    print(f"\nQuery: {query}")
    print("-" * 60)
    
    # LLM extraction
    llm_result = llm_processor.process_query(query)
    
    # Regex extraction  
    regex_result = regex_processor.process_query(query)
    
    print(f"{'Attribute':<25} {'LLM':<30} {'Regex':<30}")
    print("-" * 85)
    print(f"{'Projects':<25} {str(llm_result.potential_project_names):<30} {str(regex_result.potential_project_names):<30}")
    print(f"{'People':<25} {str(llm_result.potential_person_names):<30} {str(regex_result.potential_person_names):<30}")
    print(f"{'Dates':<25} {str(llm_result.dates):<30} {str(regex_result.dates):<30}")
    print(f"{'Technologies':<25} {str(llm_result.technologies):<30} {str(regex_result.technologies):<30}")
    print(f"{'Intent':<25} {llm_result.query_intent:<30} {regex_result.query_intent:<30}")
    print(f"{'Is Comparative':<25} {str(llm_result.is_comparative):<30} {str(regex_result.is_comparative):<30}")
    print(f"{'Confidence':<25} {llm_result.confidence:<30.2f} {regex_result.confidence:<30.2f}")

2026-03-16 10:38:09.886 | INFO     | iliad.client:__init__:135 - Initialized IliadClient with base URL: https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/chat/gpt-4o
2026-03-16 10:38:09.887 | INFO     | rag.query_processor:__init__:220 - Initialized Iliad client from environment
2026-03-16 10:38:09.888 | INFO     | rag.query_processor:__init__:225 - QueryProcessor initialized (LLM: True, Model: gpt-4o-mini-global, Few-shot: True)
2026-03-16 10:38:09.889 | INFO     | rag.query_processor:__init__:225 - QueryProcessor initialized (LLM: False, Model: gpt-4o-mini-global, Few-shot: True)
2026-03-16 10:38:09.890 | INFO     | rag.query_processor:process_query:240 - Processing query: 'Compare finder to clover methodologies'
2026-03-16 10:38:09.890 | DEBUG    | iliad.client:chat:245 - Chat request with 1 messages
2026-03-16 10:38:09.891 | DEBUG    | iliad.client:_make_request:190 - Request attempt 1/3: POST https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/chat/gpt-4o


Comparing LLM vs Regex Extraction

Query: Compare finder to clover methodologies
------------------------------------------------------------


2026-03-16 10:38:11.642 | DEBUG    | iliad.client:chat:249 - Chat response received: 507 chars
2026-03-16 10:38:11.643 | INFO     | rag.query_processor:process_query:247 - LLM extraction successful - Keywords: 3, Projects: 2, People: 0
2026-03-16 10:38:11.643 | INFO     | rag.query_processor:process_query:240 - Processing query: 'Compare finder to clover methodologies'
2026-03-16 10:38:11.644 | INFO     | rag.query_processor:process_query:258 - Using regex-based extraction


Attribute                 LLM                            Regex                         
-------------------------------------------------------------------------------------
Projects                  ['finder', 'clover']           []                            
People                    []                             []                            
Dates                     []                             []                            
Technologies              []                             []                            
Intent                    comparison                     comparison                    
Is Comparative            True                           True                          
Confidence                0.90                           0.50                          


### Test End-to-End: Query Processor -> Project Store

See if extracted project names match what the vector store retrieves.

In [ ]:
def test_query_to_retrieval(query: str, top_k: int = 5):
    """
    Test the full pipeline: Query -> Process -> Project Store -> Evaluate
    
    This helps identify if:
    1. The query processor correctly extracts the target project
    2. The project vector store returns that project in top results
    """
    print(f"{'='*80}")
    print(f"QUERY: {query}")
    print(f"{'='*80}\n")
    
    # Step 1: Process query
    processed = query_processor.process_query(query)
    extracted_projects = [p.upper() for p in processed.potential_project_names]
    
    print("STEP 1: Query Processing")
    print("-" * 40)
    print(f"  Extracted Projects: {processed.potential_project_names}")
    print(f"  Keywords: {processed.keywords}")
    print(f"  Intent: {processed.query_intent}")
    print()
    
    # Step 2: Query project store
    project_results = project_store.query_projects(query, n_results=top_k)
    retrieved_projects = [r['main_project'].upper() for r in project_results]
    
    print("STEP 2: Project Store Results")
    print("-" * 40)
    for i, r in enumerate(project_results, 1):
        proj_name = r['main_project']
        sim = r['similarity']
        match = "✓ MATCH" if proj_name.upper() in extracted_projects else ""
        print(f"  {i}. {proj_name}: {sim:.4f} {match}")
    print()
    
    # Step 3: Evaluate alignment
    print("STEP 3: Alignment Analysis")
    print("-" * 40)
    
    # Check if extracted projects appear in retrieved
    for proj in processed.potential_project_names:
        if proj.upper() in retrieved_projects:
            rank = retrieved_projects.index(proj.upper()) + 1
            print(f"  ✓ '{proj}' found at rank {rank}")
        else:
            print(f"  ✗ '{proj}' NOT in top {top_k} retrieved")
            
            # Check if it exists at all
            all_projects = [p.upper() for p in project_store.get_project_names()]
            if proj.upper() in all_projects:
                print(f"    (exists in store but ranked lower)")
            else:
                print(f"    (does NOT exist in project store)")
    
    print()
    return processed, project_results

In [ ]:
# Test queries
test_queries = [
    "What is ALFA?",
    "Describe the ATLAS project architecture",
    "How does CLOVER handle data processing?",
    "What machine learning methods does RBQM use?",
]

for q in test_queries:
    test_query_to_retrieval(q, top_k=5)
    print("\n")